# data, model

In [1]:
import modules.utils as u
gpus = u.Devices()
gpus._cuda_list_gpus()

[('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041),
 ('cuda:2', 'NVIDIA A100-SXM4-80GB', 64871),
 ('cuda:1', 'NVIDIA A100-SXM4-80GB', 57939),
 ('cuda:0', 'NVIDIA A100-SXM4-80GB', 54820),
 ('cuda:4', 'NVIDIA A100-SXM4-80GB', 52437),
 ('cuda:6', 'NVIDIA A100-SXM4-80GB', 35119),
 ('cuda:3', 'NVIDIA A100-SXM4-80GB', 8343),
 ('cuda:5', 'NVIDIA A100-SXM4-80GB', 5725)]

In [2]:
#### mv packages ####
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device, generator = u.Devices().auto_set_device(drop=['cuda:6'])

#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

data = d.Preprocessor(brca, kegg, kegg)
_dataset = d.GraphDataset(data)
_batch = d.get_toy_databatch(_dataset, generator)

# #### Device() ####
# device = cuda:7

# #### KEGG() ####
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:7)
# edge_attr                (32464, 16)              Tensor (cuda:7)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:7)

# #### TCGA() ####
# counts_path              PosixPath
# metadata_path            PosixPath
# gene_name_path           PosixPath
# metadata_complete        (1231, 93)               DataFrame
# metadata                 (1172, 2)                DataFrame
# y                        (1172,)                  Tensor (cuda:7)
# y_labels                 5                        list
# ensgv                    (60660, 3)               DataFrame
# ensg_complete            19213                    list
# cou

In [4]:
import modules.train as t
from modules.train import DataloaderMean, NBReconTrainer
import torch.nn as nn

from torch_geometric.nn import GATConv
from modules.model import Encoder, Latent, Decoder, Autoencoder
from modules.layers import AttentionSetPooling

from torch import Tensor

loader = t.Loader(
    dataset=_dataset,
    generator=generator,
    batch_size=128
)

print(data.y_labels)
dm = DataloaderMean(loader.train_loader, data.num_nodes, data.num_node_features)
# dm.plot_target_vs_global(4, data.y_labels) # plot
x_mean = dm.get_mean(4)

ae = Autoencoder(
    mask=data.pathway_index,
    num_features=data.num_node_features, # encoder
    # embed_dim=None,
    head_dim=32,
    num_heads=10,

    # layers
    node_encoder=nn.Linear, # encoder
    pooling=AttentionSetPooling, # encoder (set), latent (global)
    mlp=True, # latent

    # layer params
    hidden_dims=[128,128,128], 
    act_fn=nn.ReLU, 
    norm_fn='layer', 
    end_fn=False,

    # etc
    method='twin', # encoder, latent
    fwd=None, # latent
    expand_mlp=True, # decoder
    shared_mlp=False, # decoder
    fit_nb=False, # decoder
    log_transform=True, # autoencoder

    # nb
    x_mean=x_mean,
    learn_mu=True
    
)

trainer = NBReconTrainer(
    model=ae,
    loader=loader,
    num_epochs=30,
    loss_fn=nn.MSELoss(),
    optimizer_kwargs={'lr':5e-4},
    verbose=True,
    report_metrics=['loss','mae','rmse', 'r2'],
)

display(trainer.test_metrics)

['Basal', 'Her2', 'LumA', 'LumB', 'Solid Tissue Normal']


100%|██████████| 30/30 [01:02<00:00,  2.10s/it, Epoch 29      Train: loss=10.6195    mae=0.7421    rmse=1.1167    r2=0.8433        Val: loss=10.6424    mae=0.7498    rmse=1.1264    r2=0.8421]

Test	 loss=10.9727    mae=0.7849    rmse=1.1749    r2=0.8271



{'loss': 10.97265100479126,
 'mean': 0.111086405813694,
 'std': 1.1696597337722778,
 'mae': 0.7849259972572327,
 'mse': 1.3804423809051514,
 'rmse': 1.174922227859497,
 'r2': 0.8270875215530396}

# model output wrapper

In [4]:
import numpy as np
import torch

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import seaborn as sns
import scipy.cluster.hierarchy as sch
from matplotlib.patches import Rectangle
import matplotlib.colors as mcolors
import gc

import statsmodels.api as sm

import umap
# import sklearn.metrics
from sklearn.metrics import mean_absolute_error
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE


from torch import Tensor
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader
from typing import Literal

/home/mv18gs/miniconda3/envs/thesis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
class ModelOutput():
    def __init__(self, model, dataset, batch_size:int=64):
        self.device = device
        self.values = {}

        # initialize loader
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

        # run model
        out = {}
        x = {}
        y = {}
        batch_num = 0
        with torch.inference_mode():
            for batch in loader:
                x[batch_num] = batch.x
                y[batch_num] = batch.y
                out[batch_num] = model.get_weights(batch)
                batch_num += 1 

        # combine batches into lists
        for batch, outputs in out.items():
            for k,v in outputs.items():
                self.values.setdefault(k,[]).append(v)
        
        # combine list of tensors
        def is_tensor_list(lst):
            return isinstance(lst, list) and all(isinstance(x, torch.Tensor) for x in lst)

        self.values = {
            k: torch.cat(v, dim=0).detach().cpu().numpy() if is_tensor_list(v) else v
            for k,v in self.values.items()
        }

        # add xy
        self.values['x'] = torch.cat([i for i in x.values()], dim=0).detach().cpu().numpy()
        self.values['y'] = torch.cat([i for i in y.values()], dim=0).detach().cpu().numpy()

In [38]:
out = ModelOutput(
    model=trainer.model,
    dataset=_dataset,
)

# x recon heatmap

In [7]:
def x_recon_heatmap(
    model_output, 
    num_nodes:int, 
    class_names, 
    x_in:str, 
    x_out:str, 

    in_title:str=None, 
    out_title:str=None, 
    cbar_title:str=None, 
    row_whitespace:int = 1, 
    transform:Literal['log','exp']=None, 
    borders:bool=True, 
    figsize:tuple[int]=(14, 5)
):
        # --- Input data ---
        x_in = model_output.values[f'{x_in}'].reshape(-1, num_nodes)
        x_out = model_output.values[f'{x_out}'].reshape(-1, num_nodes)
        labels = model_output.values['y']

        if transform == 'log':
            x_in = np.log(x_in+1)
            x_out = np.log(x_out+1)
        elif transform == 'exp':
            x_in = np.exp(x_in)
            x_out = np.exp(x_out)

        in_title = in_title if in_title is not None else 'Data'
        out_title = out_title if out_title is not None else 'Model prediction'

        # --- Class metadata ---
        assert max(labels) < len(class_names), "Label index exceeds class_names"

        # --- Column clustering ---
        col_order = sch.dendrogram(sch.linkage(x_in.T, method='average'), no_plot=True)['leaves']
        x_in = x_in[:, col_order]
        x_out = x_out[:, col_order]

        # --- Cluster rows within each class ---
        unique_classes = np.unique(labels)
        row_order = []
        x_in_blocks, x_out_blocks = [], []
        block_sizes = []

        for cls in unique_classes:
            cls_indices = np.where(labels == cls)[0]
            cls_x = x_in[cls_indices]

            if cls_x.shape[0] < 2:
                cls_order = cls_indices
            else:
                row_linkage = sch.linkage(cls_x, method='average')
                cls_order = cls_indices[sch.dendrogram(row_linkage, no_plot=True)['leaves']]

            x_in_blocks.append(x_in[cls_order])
            x_out_blocks.append(x_out[cls_order])
            block_sizes.append(len(cls_order))
            row_order.extend(cls_order.tolist())

        # --- Add white space between class blocks ---
        white_row = np.full((row_whitespace, x_in.shape[1]), np.nan)

        x_in_padded, x_out_padded = [], []
        label_ticks = []
        curr_idx = 0

        for i, (xin_block, xout_block, size) in enumerate(zip(x_in_blocks, x_out_blocks, block_sizes)):
            x_in_padded.append(xin_block)
            x_out_padded.append(xout_block)
            label_ticks.append(curr_idx + size // 2)
            curr_idx += size

            if i < len(x_in_blocks) - 1:
                x_in_padded.append(white_row)
                x_out_padded.append(white_row)
                curr_idx += row_whitespace

        # --- Stack padded arrays ---
        x_in_final = np.vstack(x_in_padded)
        x_out_final = np.vstack(x_out_padded)

        # --- Plotting ---
        fig = plt.figure(figsize=figsize)
        gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.05])
        ax0 = fig.add_subplot(gs[0, 0])
        ax1 = fig.add_subplot(gs[0, 1])
        cax = fig.add_subplot(gs[0, 2])

        vmin = 0
        vmax = np.percentile(np.nan_to_num(x_in_final), 80)

        # Get actual gene index labels from col_order (before reordering)
        start_label = '0'
        end_label = str(num_nodes - 1)
        midpoint = x_in_final.shape[1] // 2

        # --- Heatmap: Data ---
        sns.heatmap(x_in_final, ax=ax0, vmin=vmin, vmax=vmax, cmap="rocket",
                    cbar=False, mask=np.isnan(x_in_final))
        ax0.set_title(in_title)
        ax0.set_yticklabels([])
        ax0.set_xticks([0, midpoint, x_in_final.shape[1]-1])
        ax0.set_xticklabels([start_label, 'Genes', end_label])
        ax0.tick_params(axis='x', rotation=0)

        # --- Heatmap: Model prediction ---
        sns.heatmap(x_out_final, ax=ax1, vmin=vmin, vmax=vmax, cmap="rocket",
                    cbar=False, mask=np.isnan(x_out_final))
        ax1.set_title(out_title)
        ax1.set_yticks([])
        ax1.set_yticklabels([])
        ax1.set_xticks([0, midpoint, x_out_final.shape[1]-1])
        ax1.set_xticklabels([start_label, 'Genes', end_label])
        ax1.tick_params(axis='x', rotation=0)

        # --- Y-axis class names ---
        ax0.set_yticks(label_ticks)
        ax0.set_yticklabels([class_names[i] for i in unique_classes], rotation=0)

        # --- Shared colorbar ---
        norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
        sm = mpl.cm.ScalarMappable(cmap="rocket", norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, cax=cax, orientation='vertical')
        if cbar_title is not None:
            cbar.set_label(cbar_title, labelpad=15)

        # --- Borders ---
        if borders:
            # # Add black rectangles around each class block
            y = 0
            for size in block_sizes:
                for ax in [ax0, ax1]:
                    rect = Rectangle((0, y), x_in.shape[1], size, linewidth=1, edgecolor='black', facecolor='none')
                    ax.add_patch(rect)
                y += size + row_whitespace

        plt.tight_layout()
        plt.show()

## *outputs*

In [8]:
# x_recon_heatmap(
#     model_output=out,
#     num_nodes=data.num_nodes,
#     class_names=data.y_labels,
#     x_in='x',
#     x_out='mu',
#     row_whitespace=10,
#     cbar_title='counts',
# )

# x_recon_heatmap(
#     model_output=out,
#     num_nodes=data.num_nodes,
#     class_names=data.y_labels,
#     x_in='x',
#     x_out='mu',
#     row_whitespace=10,
#     cbar_title='log(counts)',
#     transform='log'
# )

In [9]:
# x_recon_heatmap(
#     model_output=out,
#     num_nodes=data.num_nodes,
#     class_names=data.y_labels,
#     x_in='x',
#     x_out='x_recon',
#     row_whitespace=10,
#     cbar_title='counts',
# )

# x_recon_heatmap(
#     model_output=out,
#     num_nodes=data.num_nodes,
#     class_names=data.y_labels,
#     x_in='x',
#     x_out='x_recon',
#     row_whitespace=10,
#     cbar_title='log(counts)',
#     transform='log'
# )

In [10]:
# x_recon_heatmap(
#     model_output=out,
#     num_nodes=data.num_nodes,
#     class_names=data.y_labels,
#     x_in='lfc',
#     x_out='lfc_recon',
#     row_whitespace=10,
#     cbar_title='fold change',
#     transform='exp'
# )

# x_recon_heatmap(
#     model_output=out,
#     num_nodes=data.num_nodes,
#     class_names=data.y_labels,
#     x_in='lfc',
#     x_out='lfc_recon',
#     row_whitespace=10,
#     cbar_title='log(fold change)',
# )

# x recon scatterplot

In [11]:
def x_recon_scatter(
    model_output, 
    num_nodes:int, 
    # class_names, 
    x_in:str, 
    x_out:str, 
    
    in_title:str=None, 
    out_title:str=None, 
    transform:Literal['log','exp']=None
):
    
    # --- Input data ---
    x_in = model_output.values[f'{x_in}'].reshape(-1, num_nodes)
    x_out = model_output.values[f'{x_out}'].reshape(-1, num_nodes)
    labels = model_output.values['y']

    if transform == 'log':
        x_in = np.log(x_in+1)
        x_out = np.log(x_out+1)
    elif transform == 'exp':
        x_in = np.exp(x_in)
        x_out = np.exp(x_out)

    x = x_in
    x_recon = x_out
    x_mean = x_in.mean(axis=0)
    x_recon_mean = x_out.mean(axis=0)

    in_title = 'Mean value (per gene)' if in_title is None else in_title
    out_title = 'Reconstructed mean value (per gene)' if out_title is None else out_title

    #### plot ####
    
    # get MAE
    recon_error = np.array([mean_absolute_error(x[:, i], x_recon[:, i]) for i in range(x.shape[1])])
    error_label = "Mean absolute error"

    # Normalize error for size and color
    size_min, size_max = 20, 200
    size = size_min + (size_max - size_min) * (recon_error - recon_error.min()) / (recon_error.ptp() + 1e-8)
    norm = mcolors.Normalize(vmin=recon_error.min(), vmax=recon_error.max())
    cmap = plt.colormaps['viridis']
    colors = cmap(norm(recon_error))

    # Darkened edge colors
    dark_edge_colors = colors.copy()
    dark_edge_colors[:, :3] *= 0.5
    dark_edge_colors[:, 3] = 1

    # Fit OLS model
    X = sm.add_constant(x_mean)
    model = sm.OLS(x_recon_mean, X).fit()
    intercept, slope = model.params
    r_squared = model.rsquared
    f_statistic = model.fvalue
    p_value = model.f_pvalue

    # Prepare regression line
    x_vals = np.linspace(x_mean.min(), x_mean.max(), 100)
    regression_line = intercept + slope * x_vals

    # Begin plot
    plt.figure(figsize=(10, 8))

    # Scatter plot
    sc = plt.scatter(x_mean, x_recon_mean, c=recon_error, cmap='viridis',
                    edgecolor=dark_edge_colors, alpha=0.4, s=size)

    # y = x reference line
    plt.plot(x_vals, x_vals, color='red', linestyle='--', label='Reference line (y = x)', linewidth=1)

    # Regression line
    plt.plot(x_vals, regression_line, color='orange', linestyle='-', label=f'Regression line', linewidth=1)

    # Colorbar
    cbar = plt.colorbar(sc)
    cbar.set_label(f"{error_label}")

    # Axes labels and layout
    plt.xlabel(in_title)
    plt.ylabel(out_title)
    plt.axis("equal")

    # Add R², F, and regression equation
    reg_eq = f"$y = {intercept:.3f} + {slope:.3f}x$"
    r2_text = f"$R^2$ = {r_squared:.3f}"
    f_text = f"$F$ = {f_statistic:.1e}"
    p_text = f"$p$ = {p_value:.1e}" if p_value > 0 else "$p$ < 1e-308"

    annotation_text = f"{reg_eq}\n{r2_text}\n{f_text}\n{p_text}"

    plt.text(0.95, 0.05, annotation_text,
            transform=plt.gca().transAxes,
            fontsize=12, ha='right', va='bottom',
            bbox=dict(facecolor='white', alpha=0.7))

    # Legend for lines
    plt.legend(loc='upper left')

    plt.tight_layout()
    plt.show()

## *outputs*

In [12]:
# x_recon_scatter(
#     model_output=out,
#     num_nodes=data.num_nodes,
#     x_in='x',
#     x_out='mu',
#     transform='log'
# )

# x_recon_scatter(
#     model_output=out,
#     num_nodes=data.num_nodes,
#     x_in='x',
#     x_out='x_recon',
#     transform='log'
# )

In [13]:
# x_recon_scatter(
#     model_output=out,
#     num_nodes=data.num_nodes,
#     x_in='lfc',
#     x_out='lfc_recon',
# )

# embedding (z)

In [14]:
def embedding_scatter(
    model_output, 
    num_samples:int, 
    x:str
):
    X = model_output.values[f'{x}'].reshape(num_samples, -1)
    labels = model_output.values['y']

    # PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    # t-SNE
    tsne = TSNE(n_components=2, perplexity=30)
    X_tsne = tsne.fit_transform(X)

    # UMAP
    reducer = umap.UMAP(n_components=2)
    X_umap = reducer.fit_transform(X)

    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    methods = ['PCA', 't-SNE', 'UMAP']
    results = [X_pca, X_tsne, X_umap]

    for ax, result, name in zip(axes, results, methods):
        if labels is not None:
            scatter = ax.scatter(result[:, 0], result[:, 1], c=labels, s=10, alpha=0.8)
            ax.legend(*scatter.legend_elements(), title="Class")
        else:
            ax.scatter(result[:, 0], result[:, 1], s=10, alpha=0.8)
        ax.set_title(name)
        ax.set_xlabel("Component 1")
        ax.set_ylabel("Component 2")

    plt.tight_layout()
    plt.show()

## *outputs*

In [15]:
# embedding_scatter(
#     model_output=out,
#     num_samples=data.num_samples,
#     x='x'
# )

# embedding_scatter(
#     model_output=out,
#     num_samples=data.num_samples,
#     x='x_recon'
# )

# ## incomprehensible
# # embedding_scatter(
# #     model_output=out,
# #     num_samples=data.num_samples,
# #     x='mu'
# # )

In [16]:
# embedding_scatter(
#     model_output=out,
#     num_samples=data.num_samples,
#     x='lfc'
# )

# embedding_scatter(
#     model_output=out,
#     num_samples=data.num_samples,
#     x='lfc_recon'
# )

In [17]:
# embedding_scatter(
#     model_output=out,
#     num_samples=data.num_samples,
#     x='h'
# )

# embedding_scatter(
#     model_output=out,
#     num_samples=data.num_samples,
#     x='z'
# )

# ...

In [18]:
print(out.values.keys())

dict_keys(['x_recon', 'lfc_recon', 'lfc', 'mu', 'theta', 'h', 'z', 'z_node', 'z_set', 'attn_n2s', 'attn_n2z', 'attn_s2z', 'attn_twin', 'x', 'y'])


In [32]:
print(out.values['h'].shape)
print(out.values['z'].shape)
print(out.values['z_set'].shape)

(1172, 305, 320)
(1172, 1, 320)
(1172, 1, 320)


In [40]:
print(out.values['h'].shape)
print(out.values['z'].shape)
print(out.values['z_node'].shape)
print(out.values['z_set'].shape)

(1172, 4678, 320)
(1172, 320)
(1172, 1, 320)
(1172, 1, 320)


In [19]:
for k,v in out.values.items():
    print(k)
    print(v)
    print()


x_recon
[[6.2719897e+02]
 [2.0976965e+03]
 [3.5351309e+03]
 ...
 [7.9137276e+01]
 [2.6215912e+02]
 [2.1050489e+00]]

lfc_recon
[[-0.17908125]
 [-0.2810319 ]
 [ 0.44962755]
 ...
 [ 0.61644286]
 [ 0.15023956]
 [ 0.0430006 ]]

lfc
[[ 0.21376371]
 [-0.1981349 ]
 [ 0.10795641]
 ...
 [ 2.2241445 ]
 [ 0.4043336 ]
 [ 0.39727426]]

mu
[[7.5020392e+02]
 [2.7783901e+03]
 [2.2549387e+03]
 ...
 [4.2723160e+01]
 [2.2558841e+02]
 [2.0164490e+00]]

theta
[[1.1055889]
 [1.1093798]
 [1.1025798]
 ...
 [0.9055588]
 [1.1049938]
 [0.9278281]]

h
[[[ 1.36447638e-01  6.22613244e-02  7.96666682e-01 ...  1.31358075e+00
    1.81847239e+00  1.94411829e-01]
  [ 7.81340361e+00 -1.73530273e+01  3.61808014e+01 ...  3.86411934e+01
    6.91332703e+01  3.08301964e+01]
  [ 7.46691525e-01 -2.14131522e+00  3.30160213e+00 ...  3.79594898e+00
    6.66120625e+00  3.17740059e+00]
  ...
  [ 5.67482829e-01 -4.92561913e+00  2.04192562e+01 ...  1.93851681e+01
    3.21083946e+01  1.28016911e+01]
  [-7.70057082e-01  1.02494097e+00  

In [20]:
print(out.values['x'].shape)
print(out.values['x'].reshape(data.num_samples,-1).shape)

print(out.values['lfc'].shape)
print(out.values['lfc'].reshape(data.num_samples,-1).shape)

print(out.values['h'].shape)
print(out.values['z'].shape)


(5125156, 1)
(1172, 4373)
(5125156, 1)
(1172, 4373)
(1172, 305, 320)
(1172, 1, 320)


In [21]:
print(out.values['attn_n2s'].shape)
print(out.values['attn_s2z'].shape)
print(out.values['y'].shape)

(1172, 4373, 305)
(1172, 305, 1)
(1172,)


In [22]:
out.values['y']

array([2, 2, 2, ..., 3, 1, 2], dtype=int8)

In [23]:
# these take too long; kernel dies

# embedding_scatter(
#     model_output=out,
#     num_samples=data.num_samples,
#     x='attn_n2s'
# )

# embedding_scatter(
#     model_output=out,
#     num_samples=data.num_samples,
#     x='attn_s2z'
# )

In [24]:
print('hi')

hi
